# SE2 Data Processing Template

This notebook transforms raw data into analysis-ready data.
It includes modules for data ingestion, quality control, harmonization, feature engineering, and export.

Folders:
- Raw data: `../project_rawdata`
- Processed data: `../project_processeddata`

In [ ]:
# ========================
# 0. Import packages
# ========================
import os
from pathlib import Path
import logging

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, Polygon
import rasterio
from rasterio.mask import mask
from pyproj import CRS, Transformer

# Setup logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# ========================
# 1. Define directories
# ========================
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
external_base = project_root.parent

raw_data_dir = external_base / "project_rawdata"
proc_data_dir = external_base / "project_processeddata"
proc_data_dir.mkdir(parents=True, exist_ok=True)

logging.info(f"Raw data directory: {raw_data_dir.resolve()}")
logging.info(f"Processed data directory: {proc_data_dir.resolve()}")

## 2. Data Ingestion Functions
Load CSVs and geospatial files into pandas or geopandas.

In [ ]:
def load_csv(filename, **kwargs):
    path = raw_data_dir / filename
    logging.info(f"Loading CSV: {path}")
    return pd.read_csv(path, **kwargs)

def load_geodata(filename, **kwargs):
    path = raw_data_dir / filename
    logging.info(f"Loading GeoDataFrame: {path}")
    return gpd.read_file(path, **kwargs)

## 3. Quality Control Functions
Check for missing columns, remove NAs, detect outliers.

In [ ]:
def qc_dataframe(df, required_columns):
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        logging.warning(f"Missing required columns: {missing_cols}")
    df = df.dropna(subset=required_columns)
    return df

def detect_outliers(df, column, method='iqr'):
    if method == 'iqr':
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        outliers = df[(df[column] < Q1 - 1.5*IQR) | (df[column] > Q3 + 1.5*IQR)]
        logging.info(f"Detected {len(outliers)} outliers in column {column}")
        return outliers
    return pd.DataFrame()

## 4. Harmonization Functions
Reproject geospatial data and resample rasters.

In [ ]:
def reproject_gdf(gdf, target_crs="EPSG:4326"):
    gdf = gdf.to_crs(target_crs)
    logging.info(f"Reprojected to {target_crs}")
    return gdf

def resample_raster(src_path, dst_path, target_resolution):
    with rasterio.open(src_path) as src:
        data = src.read(
            out_shape=(
                src.count,
                int(src.height * src.res[0] / target_resolution),
                int(src.width * src.res[1] / target_resolution)
            ),
            resampling=rasterio.enums.Resampling.bilinear
        )
        transform = src.transform * src.transform.scale(
            (src.width / data.shape[-1]),
            (src.height / data.shape[-2])
        )
        kwargs = src.meta.copy()
        kwargs.update({"height": data.shape[1], "width": data.shape[2], "transform": transform})
        with rasterio.open(dst_path, "w", **kwargs) as dst:
            dst.write(data)
    logging.info(f"Raster resampled to {target_resolution} and saved at {dst_path}")

## 5. Feature Engineering Functions
Example: compute cumulative and rolling mean for precipitation.

In [ ]:
def create_features(df):
    if 'precipitation_mm' in df.columns:
        df['precip_cumsum'] = df['precipitation_mm'].cumsum()
        df['precip_mean'] = df['precipitation_mm'].rolling(window=30, min_periods=1).mean()
        logging.info("Created precipitation cumulative and rolling mean features")
    return df

## 6. Export Functions
Save analysis-ready CSV and GeoDataFrame files.

In [ ]:
def save_dataframe(df, filename):
    path = proc_data_dir / filename
    df.to_csv(path, index=False)
    logging.info(f"Saved processed data to {path}")

def save_geodata(gdf, filename):
    path = proc_data_dir / filename
    gdf.to_file(path)
    logging.info(f"Saved processed GeoDataFrame to {path}")

## 7. Example Workflow
Load raw CSV and GeoData, QC, feature engineering, and export.

In [ ]:
# Load CSV
df = load_csv("precipitation.csv")

# Quality control
df = qc_dataframe(df, required_columns=['date', 'precipitation_mm'])
detect_outliers(df, column='precipitation_mm')

# Feature engineering
df = create_features(df)

# Save analysis-ready CSV
save_dataframe(df, "precipitation_processed.csv")

# Load geospatial data
gdf = load_geodata("stations.geojson")
gdf = reproject_gdf(gdf, target_crs="EPSG:4326")
save_geodata(gdf, "stations_processed.geojson")